# KTH LRCN Preprocessing Replica

This notebook is a replica of the main KTH LRCN workflow, written in a more explanatory form. It follows the same idea as the original project file: extract frames from a KTH action video, sample them over time, and prepare them for an LRCN-style deep learning model.

The main goal is to understand the preprocessing stage clearly before building the full training pipeline.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV version:", cv2.__version__)

In [ ]:
# Define a fixed temporal sampling window
num_frames = 16
frame_step = 6
sample_indices = np.arange(0, num_frames * frame_step, frame_step)

print("Sample indices:", sample_indices)

In [ ]:
# Choose one video from the walking action folder
base_dir = Path(r"C:\NNDL Lab Proj\KTH")
video_path = base_dir / "walking" / "person01_walking_d1_uncomp.avi"

print("Video path:", video_path)
print("Exists:", video_path.exists())

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {video_path}")

frames = []
for index in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(index))
    ret, frame = cap.read()
    if ret:
        frames.append(frame)
    else:
        print(f"Frame at index {index} could not be read.")

cap.release()

print("Number of frames loaded:", len(frames))
if frames:
    print("Frame shape:", frames[0].shape)

In [ ]:
# Resize and normalize the sampled frames
TARGET_HEIGHT = 224
TARGET_WIDTH = 224

processed_frames = []
for frame in frames:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_resized = cv2.resize(frame_rgb, (TARGET_WIDTH, TARGET_HEIGHT))
    frame_normalized = frame_resized.astype(np.float32) / 255.0
    processed_frames.append(frame_normalized)

clip = np.stack(processed_frames, axis=0)
print("Processed clip shape:", clip.shape)

In [ ]:
# Visualize the selected frames from the clip
plt.figure(figsize=(12, 8))
for i, frame in enumerate(processed_frames[:16]):
    plt.subplot(4, 4, i + 1)
    plt.imshow(frame)
    plt.title(f"Frame {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Explanation

This preprocessing stage follows the same logic used in many video action recognition pipelines:

- sample a fixed number of frames from the video
- convert the frames to a standard size
- normalize pixel values for stable training
- stack the frames into one temporal clip

For LRCN, the important idea is that the model learns from temporal patterns across consecutive frames, not from a single image. Therefore, it is necessary to keep the sequence consistent in both length and size.

## Why this is useful

- It allows the model to see motion over time.
- It reduces computational cost compared with using every frame.
- It gives a simple and reproducible preprocessing structure for all KTH action videos.

## Next step

The next step would be to build a generalized dataset loader that reads all videos from the action folders, assigns labels, and returns a batch of clips ready for training.